# 04 Baseline 与问题一预测

本 Notebook 复现问题一的移动平均、同星期均值和简单指数平滑模型。

## 代码说明：读取数据并构造预测序列

这段代码读取日销量面板并构造门店、商品、门店-商品三个预测层级。

In [ ]:
import pandas as pd
from src.config import PROJECT_ROOT
from src.models import prepare_series_table, validate_univariate_models, forecast_future_univariate
from src.evaluation import metrics_by_group, compare_models

daily = pd.read_csv(PROJECT_ROOT / 'data/processed/daily_store_product_sales.csv', parse_dates=['date'])
daily['target_sales'] = daily['positive_sales'].astype(float)
store_series = prepare_series_table(daily, ['store_id','store_name'], 'target_sales')
store_series.head()


输出怎么看：每行是一家门店某一天的销量，这是时间序列验证的基本单位。

## 代码说明：滚动验证

这段代码用 2022-03-01 至 2022-03-31 做验证期。每个验证日只使用该日之前的数据，避免时间泄露。

In [ ]:
validation_start = '2022-03-01'
valid_store = validate_univariate_models(store_series, ['store_id','store_name'], 'target_sales', validation_start)
valid_store['level'] = 'store'
metrics = metrics_by_group(valid_store, ['level','model','model_label'])
compare_models(metrics, 'level')


输出怎么看：WAPE 越小，总体预测相对误差越小；模型选择优先看 WAPE，同时参考 MAE 和 RMSE。

## 代码说明：未来 7 天预测

这段代码用验证期表现最好的模型预测 2022-04-01 至 2022-04-07。

In [ ]:
best_model = compare_models(metrics, 'level').iloc[0]['model']
future_dates = pd.date_range(daily['date'].max() + pd.Timedelta(days=1), periods=7)
future_store = forecast_future_univariate(store_series, ['store_id','store_name'], 'target_sales', future_dates, best_model)
future_store.groupby(['store_id','store_name','model_label'], as_index=False).agg(predicted_7day_sales=('prediction','sum'))


输出怎么看：预测结果是未来 7 天总销量，不包含天气和促销情景，适合作为问题一 baseline。